In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as sts
from statsmodels.formula.api import ols
import statsmodels.api as sm

In [2]:
data = pd.read_csv('../data/final.csv')
data['log_scored_by'] = np.log10(data['scored_by'])
data['sqrt_chapters'] = np.power(data['chapters'], 0.5)
columns = ['A', 'log_scored_by', 'volumes', 'sqrt_chapters',  'has Romance', 'has Comedy', 'has Hentai', 'has Fantasy',
       'has Boys Love', 'has School', 'has Historical', 'has Harem',
       'has Psychological', 'has Isekai']

data = data.sample(frac=1)
print(len(data))
tests = data.head(4)
print(len(tests))
data = data.tail(- len(tests))
print(len(data))

180
4
176


In [3]:
def load_YX(data, columns):
  X = data[columns[1:]].to_numpy()

  X = np.hstack((np.ones((X.shape[0], 1)), X))
  Y = data[['score']].to_numpy()
  return Y, X

Y, X = load_YX(data, columns)

In [4]:
def solve_multi_linear_model(Y, X):
  B = np.linalg.inv(X.T @ X) @ X.T @ Y
  e = Y - X @ B
  return B, e

B, e = solve_multi_linear_model(Y, X)


In [5]:
def get_corr(columns):
  return data[['score'] + columns[1:]].corr().round(2)

get_corr(columns)

,score,log_scored_by,volumes,sqrt_chapters,has Romance,has Comedy,has Hentai,has Fantasy,has Boys Love,has School,has Historical,has Harem,has Psychological,has Isekai
score,1.00,0.46,0.27,0.38,0.05,0.01,-0.12,0.00,-0.14,0.11,0.00,-0.05,0.01,0.06
log_scored_by,0.46,1.00,0.35,0.34,0.06,0.17,-0.14,-0.03,-0.16,0.20,-0.05,-0.08,0.02,-0.05
volumes,0.27,0.35,1.00,0.87,0.19,0.17,-0.12,0.21,-0.24,0.21,-0.01,0.10,-0.04,-0.03
sqrt_chapters,0.38,0.34,0.87,1.00,0.14,0.16,-0.09,0.21,-0.34,0.25,0.03,0.10,-0.03,0.08
has Romance,0.05,0.06,0.19,0.14,1.00,0.19,-0.15,-0.13,-0.36,0.07,0.04,0.06,0.13,-0.05
has Comedy,0.01,0.17,0.17,0.16,0.19,1.00,-0.12,0.05,-0.08,0.06,-0.16,-0.04,-0.04,-0.05
has Hentai,-0.12,-0.14,-0.12,-0.09,-0.15,-0.12,1.00,-0.05,-0.11,-0.07,-0.06,0.28,-0.05,-0.02
has Fantasy,0.00,-0.03,0.21,0.21,-0.13,0.05,-0.05,1.00,-0.28,-0.05,0.21,0.04,-0.14,0.13
has Boys Love,-0.14,-0.16,-0.24,-0.34,-0.36,-0.08,-0.11,-0.28,1.00,-0.03,-0.09,-0.10,-0.13,-0.04
has School,0.11,0.20,0.21,0.25,0.07,0.06,-0.07,-0.05,-0.03,1.00,-0.10,0.12,-0.09,-0.03


In [6]:
def standart_coef(Y, X, B):
  sigma_y = np.sqrt(np.sum((Y - np.mean(Y)) ** 2) / Y.shape[0])
  sigma_x = np.sqrt(np.array([np.sum((X - np.mean(X, axis=0)) ** 2, axis=0)]) / X.shape[0]) 
  return (B[1:] * sigma_y) / sigma_x.T[1:]

def print_cool_B(B):
  ret = f"y = {B[0][0]:.4f} "
  for i in range(1, B.shape[0]):
    ret += f"+ {B[i][0]:.4f} x_{i} "
  return ret

def print_cool_B_norm(B_norm):
  ret = f"t_y = {B_norm[0][0]:.2f} t_x_1 "
  for i in range(1, B_norm.shape[0]):
    ret += f"+ {B_norm[i][0]:.2f} t_x_{i + 1} "
  return ret

B_norm = standart_coef(Y, X, B)
print(print_cool_B(B))
print(print_cool_B_norm(B_norm))


y = 5.7839 + 0.4456 x_1 + -0.0385 x_2 + 0.0874 x_3 + 0.0328 x_4 + -0.1406 x_5 + -0.1681 x_6 + -0.0279 x_7 + 0.0162 x_8 + -0.0676 x_9 + -0.0457 x_10 + -0.0551 x_11 + -0.0353 x_12 + 0.1610 x_13 
t_y = 0.49 t_x_1 + -0.00 t_x_2 + 0.01 t_x_3 + 0.04 t_x_4 + -0.18 t_x_5 + -0.49 t_x_6 + -0.04 t_x_7 + 0.02 t_x_8 + -0.12 t_x_9 + -0.10 t_x_10 + -0.17 t_x_11 + -0.09 t_x_12 + 1.23 t_x_13 


In [7]:
def pretty_table(data):
  p = []
  for line in data:
    while len(p) < len(line):
      p.append(0)
    for i in range(len(line)):
      p[i] = max(p[i], len(line[i]))
  
  def print_line(line):
    def add_spaces(s, c):
      return (" " * c) + s
    ret = add_spaces(line[0], p[0] - len(line[0]))
    for i in range(1, len(line)):
      ret += " │ " + add_spaces(line[i], p[i] - len(line[i]))
    for i in range(len(line), len(p)):
      ret += " │ " + p[i] * " "
    return ret

  line_sep = "─" * p[0]
  for i in range(1, len(p)):
    line_sep += "─┼─" + "─" * p[i]

  ret = ""
  ret += print_line(data[0]) + "\n" + line_sep + '\n'
  for i in range(1, len(data)):
    ret += print_line(data[i]) + "\n"
  return ret


In [8]:
def t_Student_criterion(Y, X, B, e, columns, is_print = True, alpha = 0.05):
  dfd = X.shape[0] - X.shape[1]  # Степени свободы знаменателя (внутригрупповые)

  S_2 = e.T @ e / (dfd) # стандартная ошибка в квадрате
  Cov_B = S_2 * np.linalg.inv(X.T @ X)

  SE = np.sqrt(S_2 * np.array([[Cov_B[i][i]] for i in range(Cov_B.shape[1]) ]))

  t_fact = B / SE

  t_table = sts.t.ppf(1 - alpha / 2, dfd)


  table = [["Название критерия", "B", "t_fact", "B!=0"]]

  ret = []

  for i in range(len(columns)):
    table.append([
      columns[i],
      f"{B[i][0]:.5f}",
      f"{t_fact[i][0]:.3f}",
      "+" if np.abs(t_fact[i][0]) > t_table else "-"
    ])
    ret.append(np.abs(t_fact[i][0]) > t_table)
  
  if is_print:
    print(f"t_table = {t_table:.2f}")
    print()
    print(pretty_table(table))
  return ret, t_fact

t_Student_criterion(Y, X, B, e, columns)
pass

t_table = 1.97

Название критерия │        B │ t_fact │ B!=0
──────────────────┼──────────┼────────┼─────
                A │  5.78388 │ 49.129 │    +
    log_scored_by │  0.44556 │ 11.117 │    +
          volumes │ -0.03854 │ -4.461 │    +
    sqrt_chapters │  0.08735 │  7.412 │    +
      has Romance │  0.03285 │  0.688 │    -
       has Comedy │ -0.14062 │ -3.107 │    +
       has Hentai │ -0.16806 │ -1.575 │    -
      has Fantasy │ -0.02790 │ -0.540 │    -
    has Boys Love │  0.01623 │  0.282 │    -
       has School │ -0.06765 │ -1.093 │    -
   has Historical │ -0.04572 │ -0.576 │    -
        has Harem │ -0.05515 │ -0.496 │    -
has Psychological │ -0.03534 │ -0.418 │    -
       has Isekai │  0.16100 │  0.619 │    -



In [9]:
def F_criterion(Y, X, B, e, alpha = 0.05):
  SS_all = np.sum((Y - np.mean(Y)) ** 2)
  SS_R = np.sum((X @ B - np.mean(Y)) ** 2)
  SS_e = np.sum(e ** 2)
  R_2 = SS_R / SS_all
  r = np.sqrt(R_2)

  R_2_norm = 1 - (1 - R_2) * (X.shape[0] - 1) / (X.shape[0] - X.shape[1])
  r_norm =  np.sqrt(R_2_norm)

  dfn = X.shape[1] - 1           # Степени свободы числителя (межгрупповые)
  dfd = X.shape[0] - X.shape[1]  # Степени свободы знаменателя (внутригрупповые)

  MS_R = SS_R / (dfn)
  MS_e = SS_e / (dfd)

  F_fact = MS_R / MS_e

  F_table = sts.f.ppf(1 - alpha, dfn, dfd)


  print(f"SS_all = {SS_all:.3f}, SS_R = {SS_R:.3f}, SS_e = {SS_e:.3f} ")
  print(f"Проверка: SS_R + SS_e = {SS_R + SS_e:.3f}")
  print(f"общий коэффициент детерминации: {R_2:.3f}")
  print(f"общий коэффициент корреляции: {r:.3f}")
  print(f"скорректированный коэффициент детерминации: {R_2_norm:.3f}")
  print(f"скорректированный коэффициент корреляции: {r_norm:.3f}")
  
  print(f"""тестнота связи: {
    "не наблюдается" if r_norm < 0.1 else 
    "слабая" if r_norm < 0.3 else
    "умеренная" if r_norm < 0.5 else
    "заметная" if r_norm < 0.7 else
    "высокая" if r_norm < 0.9 else
    "весьма высокая"
  }""")
  print()
  table = [
    ["", "df", "SS", "MS", "F", "значимость F"],
    ["Регрессия", f"{dfn}", f"{SS_R:.3f}", f"{MS_R:.3f}", f"{F_fact:.3f}", f"{F_table:.3f}"],
    ["Остаток", f"{dfd}", f"{SS_e:.3f}", f"{MS_e:.3f}"],
    ["Итого", f"{dfn + dfd}", f"{SS_all:.3f}"]
  ]
  print(pretty_table(table))
  print()
  print("Линейность наблюдается" if F_fact > F_table else "Нет оснований предпологать линейность")

F_criterion(Y, X, B, e)

SS_all = 58.302, SS_R = 18.000, SS_e = 40.302 
Проверка: SS_R + SS_e = 58.302
общий коэффициент детерминации: 0.309
общий коэффициент корреляции: 0.556
скорректированный коэффициент детерминации: 0.253
скорректированный коэффициент корреляции: 0.503
тестнота связи: заметная

          │  df │     SS │    MS │     F │ значимость F
──────────┼─────┼────────┼───────┼───────┼─────────────
Регрессия │  13 │ 18.000 │ 1.385 │ 5.565 │        1.781
  Остаток │ 162 │ 40.302 │ 0.249 │       │             
    Итого │ 175 │ 58.302 │       │       │             


Линейность наблюдается


In [10]:
columns_clear = columns.copy()

while len(columns_clear) > 0:
  Y, X = load_YX(data, columns_clear)
  B, e = solve_multi_linear_model(Y, X)
  is_goods, t_fact = t_Student_criterion(Y, X, B, e, columns_clear, False)
  is_out = True
  for is_good in is_goods:
    is_out = is_good and is_out
  if is_out:
    break
  i_del = np.argmin(np.abs(t_fact.T[0]))
  columns_clear.pop(i_del)

Y, X = load_YX(data, columns_clear)
B, e = solve_multi_linear_model(Y, X)

B_norm = standart_coef(Y, X, B)
print(print_cool_B(B))
print(print_cool_B_norm(B_norm))
print()
t_Student_criterion(Y, X, B, e, columns_clear)
F_criterion(Y, X, B, e)


y = 5.7965 + 0.4421 x_1 + -0.0383 x_2 + 0.0847 x_3 + -0.1318 x_4 + -0.1847 x_5 
t_y = 0.49 t_x_1 + -0.00 t_x_2 + 0.01 t_x_3 + -0.17 t_x_4 + -0.54 t_x_5 

t_table = 1.97

Название критерия │        B │ t_fact │ B!=0
──────────────────┼──────────┼────────┼─────
                A │  5.79649 │ 59.475 │    +
    log_scored_by │  0.44212 │ 11.916 │    +
          volumes │ -0.03827 │ -4.985 │    +
    sqrt_chapters │  0.08467 │  8.285 │    +
       has Comedy │ -0.13177 │ -3.160 │    +
       has Hentai │ -0.18472 │ -1.975 │    +

SS_all = 58.302, SS_R = 17.800, SS_e = 40.502 
Проверка: SS_R + SS_e = 58.302
общий коэффициент детерминации: 0.305
общий коэффициент корреляции: 0.553
скорректированный коэффициент детерминации: 0.285
скорректированный коэффициент корреляции: 0.534
тестнота связи: заметная

          │  df │     SS │    MS │      F │ значимость F
──────────┼─────┼────────┼───────┼────────┼─────────────
Регрессия │   5 │ 17.800 │ 3.560 │ 14.942 │        2.267
  Остаток │ 170 │ 40.5

In [11]:
corr = get_corr(columns_clear)
print(corr)

               score  log_scored_by  volumes  sqrt_chapters  has Comedy  \
score           1.00           0.46     0.27           0.38        0.01   
log_scored_by   0.46           1.00     0.35           0.34        0.17   
volumes         0.27           0.35     1.00           0.87        0.17   
sqrt_chapters   0.38           0.34     0.87           1.00        0.16   
has Comedy      0.01           0.17     0.17           0.16        1.00   
has Hentai     -0.12          -0.14    -0.12          -0.09       -0.12   

               has Hentai  
score               -0.12  
log_scored_by       -0.14  
volumes             -0.12  
sqrt_chapters       -0.09  
has Comedy          -0.12  
has Hentai           1.00  


In [12]:
columns_very_clear = columns_clear.copy()
try:
  columns_very_clear.remove("volumes")
  pass
except:
  pass

Y, X = load_YX(data, columns_very_clear)
B, e = solve_multi_linear_model(Y, X)

B_norm = standart_coef(Y, X, B)
print(print_cool_B(B))
print(print_cool_B_norm(B_norm))
print()
t_Student_criterion(Y, X, B, e, columns_very_clear)
F_criterion(Y, X, B, e)

y = 5.8956 + 0.4248 x_1 + 0.0415 x_2 + -0.1411 x_3 + -0.1540 x_4 
t_y = 0.47 t_x_1 + 0.01 t_x_2 + -0.18 t_x_3 + -0.45 t_x_4 

t_table = 1.97

Название критерия │        B │ t_fact │ B!=0
──────────────────┼──────────┼────────┼─────
                A │  5.89560 │ 60.063 │    +
    log_scored_by │  0.42477 │ 11.177 │    +
    sqrt_chapters │  0.04146 │  7.444 │    +
       has Comedy │ -0.14109 │ -3.293 │    +
       has Hentai │ -0.15396 │ -1.603 │    -

SS_all = 58.302, SS_R = 16.389, SS_e = 41.913 
Проверка: SS_R + SS_e = 58.302
общий коэффициент детерминации: 0.281
общий коэффициент корреляции: 0.530
скорректированный коэффициент детерминации: 0.264
скорректированный коэффициент корреляции: 0.514
тестнота связи: заметная

          │  df │     SS │    MS │      F │ значимость F
──────────┼─────┼────────┼───────┼────────┼─────────────
Регрессия │   4 │ 16.389 │ 4.097 │ 16.716 │        2.425
  Остаток │ 171 │ 41.913 │ 0.245 │        │             
    Итого │ 175 │ 58.302 │       │    

In [13]:
corr = get_corr(columns_very_clear)
print(corr)

               score  log_scored_by  sqrt_chapters  has Comedy  has Hentai
score           1.00           0.46           0.38        0.01       -0.12
log_scored_by   0.46           1.00           0.34        0.17       -0.14
sqrt_chapters   0.38           0.34           1.00        0.16       -0.09
has Comedy      0.01           0.17           0.16        1.00       -0.12
has Hentai     -0.12          -0.14          -0.09       -0.12        1.00


In [14]:
# Прогноз

test_Y, test_X = load_YX(tests, columns_very_clear)
test_Y_pred = test_X @ B
test_e = test_Y - test_Y_pred
test_e_2 = test_e ** 2
sum_e_2 = np.sum(test_e_2)
sum_e = np.sqrt(sum_e_2 / len(tests))

table = [["Реальная оценка", "Предсказаная оценка", "Ошибка", "Квадрат ошибки"]]
for i in range(test_Y.shape[0]):
  table.append([
    f"{test_Y[i][0]:.2f}",
    f"{test_Y_pred[i][0]:.2f}",
    f"{test_e[i][0]:.2f}",
    f"{test_e_2[i][0]:.2f}",
  ])
table.append([
  "", 
  "Сумма", "--",
  f"{sum_e_2:.2f}",
])
table.append([
  "", 
  "Нормирования ошибка", "--",
  f"!! {sum_e:.2f} !!",
])
print(pretty_table(table))



Реальная оценка │ Предсказаная оценка │ Ошибка │ Квадрат ошибки
────────────────┼─────────────────────┼────────┼───────────────
           6.50 │                6.80 │  -0.30 │           0.09
           6.41 │                7.05 │  -0.64 │           0.41
           6.87 │                6.88 │  -0.01 │           0.00
           7.50 │                6.85 │   0.65 │           0.42
                │               Сумма │     -- │           0.92
                │ Нормирования ошибка │     -- │     !! 0.48 !!

